In [ ]:
print("Jay Ganesh")

Jay Ganesh


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter


d:\ProjectAarya\GEN AI\Code\Ch22\ch22env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
loader=TextLoader("speech.txt")
doc=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=30,chunk_overlap=0)
docwithsplit=text_splitter.split_documents(doc)


Created a chunk of size 470, which is longer than the specified 30
Created a chunk of size 347, which is longer than the specified 30
Created a chunk of size 668, which is longer than the specified 30
Created a chunk of size 982, which is longer than the specified 30
Created a chunk of size 789, which is longer than the specified 30


In [7]:
docwithsplit

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.'),
 Document(metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.'),
 Document(metadata={'source': 'speech.txt'}, page_content

In [9]:
# Initialize local embeddings using Ollama's lightweight TinyLlama model
# This converts text chunks into dense vector representations for semantic search
embedding = OllamaEmbeddings(model="tinyllama:latest")

# Create FAISS vector store from pre-split documents (docwithsplit)
# Automatically embeds all document chunks and builds efficient similarity search index
db = FAISS.from_documents(docwithsplit, embedding)

# 'db' is now ready for semantic search, retrieval, or RAG chains


In [10]:
query="Where does the speaker describe the desired outcome of the war?"

q1=db.similarity_search(query)

In [11]:
q1

[Document(id='309451dc-a6ee-4643-9506-1388dee3f84c', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='a1f4b4ba-b022-40aa-8fca-6e6f71f0ae03', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to sha

In [12]:
q1[0].page_content

'It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### Now trying as a retriever
We can also convert the vectorstore into a Retriver class. This allows us to easily use it in other LangChain methods, which largely work with retrievers.

In [14]:
retriever=db.as_retriever()
doc2=retriever.invoke(query)

In [16]:
doc2[0].page_content

'It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

#### Similarity Search with Score
##### There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better 

In [17]:
doc_and_score=db.similarity_search_with_score(query)

In [18]:
doc_and_score

[(Document(id='309451dc-a6ee-4643-9506-1388dee3f84c', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(6489.285)),
 (Document(id='a1f4b4ba-b022-40aa-8fca-6e6f71f0ae03', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves bu

In [ ]:
### The more the score is nearer to 0 more it is rich in data
## We can also perform the same thing but using embedding vector of the query
embedding_vector=embedding.embed_query(query)


In [20]:
db.similarity_search_by_vector(embedding_vector)

[Document(id='309451dc-a6ee-4643-9506-1388dee3f84c', metadata={'source': 'speech.txt'}, page_content='It will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='a1f4b4ba-b022-40aa-8fca-6e6f71f0ae03', metadata={'source': 'speech.txt'}, page_content='Just because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to sha

#### Let's now understand how to sore the vector db into our local file

In [21]:
db.save_local("Faiss_index")

In [24]:
new_db=FAISS.load_local("Faiss_index",embedding)

ValueError: The de-serialization relies loading a pickle file. Pickle files can be modified to deliver a malicious payload that results in execution of arbitrary code on your machine.You will need to set `allow_dangerous_deserialization` to `True` to enable deserialization. If you do this, make sure that you trust the source of the data. For example, if you are loading a file that you created, and know that no one else has modified the file, then this is safe to do. Do not set this to `True` if you are loading a file from an untrusted source (e.g., some random site on the internet.).

In [25]:
### We can avoid such error but allow dangerous deserialization as True
new_db=FAISS.load_local("Faiss_index",embedding,allow_dangerous_deserialization=True)